In [2]:
from tempfile import template

import pandas as pd
import plotly.express as px

import  plotly.graph_objects as go
from unicodedata import category

In [3]:
df=pd.read_csv('../data/football_data_cleaned.csv')
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1352 entries, 0 to 1351
Data columns (total 51 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   Player   1352 non-null   object 
 1   Nation   1352 non-null   object 
 2   Pos      1352 non-null   object 
 3   Squad    1352 non-null   object 
 4   Comp     1352 non-null   object 
 5   Age      1352 non-null   float64
 6   MP       1352 non-null   int64  
 7   Starts   1352 non-null   int64  
 8   Min      1352 non-null   int64  
 9   90s      1352 non-null   float64
 10  Gls      1352 non-null   int64  
 11  Ast      1352 non-null   int64  
 12  G+A      1352 non-null   int64  
 13  G-PK     1352 non-null   int64  
 14  PK       1352 non-null   int64  
 15  PKatt    1352 non-null   int64  
 16  CrdY     1352 non-null   int64  
 17  CrdR     1352 non-null   int64  
 18  G+A-PK   1352 non-null   float64
 19  Sh       1352 non-null   int64  
 20  SoT      1352 non-null   int64  
 21  SoT%     1352 

In [16]:
df_attacker=df[df['Pos'].str.contains('FW|MF',na=False)]
fig=px.scatter(
    df_attacker,
    x='Ast_90',
    y='Gls_90',
    color='Comp',
    hover_name='Player',
    hover_data=['Squad','Age','Min'],
    title='Evaluation Offensive : Buts vs Passes Décisives (par 90  min)',
    labels={
        "Ast_90":"Assist (90 min)",
        "Gls_90":"Goal (90 min)",
        "Comp":"Championnat"
    },
    template='plotly_dark'


)
fig.update_traces(marker=dict(size=10,opacity=0.8,line=dict(width=1,color='DarkSlateGrey')))
fig.show()

In [9]:
categories = ['Gls_90', 'Ast_90', 'Sh_90', 'SoT_90', 'Crs_90', 'Int_90']
joueur_1="Ousmane Dembélé"
joueur_2="Lamine Yamal"

joueur_1_data=df[df['Player']==joueur_1]
joueur_2_data=df[df['Player']==joueur_2]


if joueur_1_data.empty or joueur_2_data.empty:
    print("un joueur n'a pas trouvee")
else:
    fig=go.Figure()

    fig.add_trace(
        go.Scatterpolar(
            r=joueur_1_data[categories].values[0],
            theta=categories,
            fill='toself',
            name=joueur_1,
            line=dict(color='#00D9FF', width=3),  # Cyan plus vif avec bordure épaisse
            fillcolor='rgba(0, 217, 255, 0.3)',  # Remplissage semi-transparent
            marker=dict(size=8, color='#00D9FF', line=dict(width=2, color='white'))
        )
    )
    fig.add_trace(
        go.Scatterpolar(
            r=joueur_2_data[categories].values[0],
            theta=categories,
            fill='toself',
            name=joueur_2,
            line=dict(color='#FF6B35', width=3),  # Orange plus vif avec bordure épaisse
            fillcolor='rgba(255, 107, 53, 0.3)',  # Remplissage semi-transparent
            marker=dict(size=8, color='#FF6B35', line=dict(width=2, color='white'))
        )
    )
    fig.update_layout(
        polar=dict(
            bgcolor='rgba(20, 20, 20, 0.8)',  # Fond polar légèrement transparent
            radialaxis=dict(
                visible=True,
                range=[0, max(joueur_1_data[categories].max().max(),joueur_2_data[categories].max().max())+0.5],
                gridcolor='rgba(100, 100, 100, 0.5)',  # Grille plus visible
                tickfont=dict(size=11, color='#E0E0E0')
            ),
            angularaxis=dict(
                gridcolor='rgba(100, 100, 100, 0.5)',
                tickfont=dict(size=12, color='#E0E0E0', family='Arial Black')
            )
        ),
        showlegend=True,
        legend=dict(
            x=1.1,
            y=1,
            font=dict(size=13, color='white'),
            bgcolor='rgba(0, 0, 0, 0.5)',
            bordercolor='#E0E0E0',
            borderwidth=1
        ),
        title=dict(
            text=f"Evaluation Offensive : {joueur_1} vs {joueur_2}",
            font=dict(size=18, color='white', family='Arial Black'),
            x=0.5,
            xanchor='center'
        ),
        template='plotly_dark',
        paper_bgcolor='#0A0A0A',
        font=dict(color='white', size=12),
        height=700,
        width=900,
        margin=dict(l=80, r=80, t=100, b=80)
    )
    fig.show()

In [13]:

df_finishers = df[(df['Pos'].str.contains('FW|MF', na=False)) & (df['Sh_90'] > 1.0)]


fig = px.scatter(
    df_finishers,
    x='Sh_90',
    y='Gls_90',
    color='Comp',
    hover_name='Player',

    hover_data=['Squad', 'Age', 'Gls', 'Sh', 'G/Sh'],
    title='Analyse de la Finition : Volume de Tirs (Sh_90) vs Buts (Gls_90)',
    labels={
        'Sh_90': 'Tirs tentés (par 90 min)',
        'Gls_90': 'Buts marqués (par 90 min)',
        'Comp': 'Championnat'
    },
    template='plotly_dark'
)

fig.update_traces(marker=dict(size=9, opacity=0.8, line=dict(width=0.5, color='white')))

fig.show()

In [15]:
df_midfielders = df[df['Pos'].str.contains('MF', na=False)]

top_15_creators = df_midfielders.sort_values(by='Ast_90', ascending=False).head(15)

fig = px.bar(
    top_15_creators,
    x='Ast_90',
    y='Player',
    orientation='h',
    color='Comp',
    hover_data=['Squad', 'Age', 'Ast', 'Min'], # Infos utiles au survol
    title='Les 15 Meilleurs Milieux Créateurs d\'Europe (Passes Décisives / 90 min)',
    labels={
        'Ast_90': 'Passes Décisives (par 90 min)',
        'Player': 'Joueur',
        'Comp': 'Championnat'
    },
    template='plotly_dark'
)


fig.update_layout(yaxis={'categoryorder':'total ascending'})

fig.show()

In [17]:

df_impact = df[df['Min'] >= 900]

fig = px.scatter(
    df_impact,
    x='+/-90',                 # Performance globale de l'équipe avec lui
    y='On-Off',                # Impact réel du joueur sur son équipe
    color='Comp',              # Distinction par championnat
    hover_name='Player',
    hover_data=['Squad', 'Pos', 'Age', 'Min%'],
    title='Analyse de l\'Influence : Performance Équipe ($+/-90$) vs Importance du Joueur (On-Off)',
    labels={
        '+/-90': 'Différentiel de buts de l\'équipe (par 90 min)',
        'On-Off': 'Impact Net (Performance avec lui - sans lui)',
        'Comp': 'Championnat'
    },
    template='plotly_dark'
)
fig.add_hline(y=0, line_dash="dash", line_color="gray")

fig.show()